# Fase 2: Pelatihan Model LSTM-Autoencoder
## Proyek Skripsi: Deteksi Anomali Gerakan Rehabilitasi Tangan
### Menggunakan Autoencoder Berbasis Data Pose Estimation (MediaPipe)

---
**CARA PAKAI:**
1. Upload file `dataset_normal.csv` ke Colab (klik ikon Folder di panel kiri → Upload)
2. Klik **Runtime → Run All** atau jalankan sel satu per satu dari atas ke bawah
3. Catat nilai **THRESHOLD** dari output Sel 9
4. Download ketiga file hasil: `otak_Rehabilitasi.keras`, `scaler_Rehabilitasi.pkl`, `threshold_value.txt`
---

In [ ]:
# SEL 1: Install Library
# Jalankan sel ini terlebih dahulu
!pip install tensorflow numpy pandas matplotlib scikit-learn seaborn joblib -q
print('Semua library berhasil diinstall!')

In [ ]:
# SEL 2: Import Library
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, RepeatVector, TimeDistributed, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

print(f'TensorFlow version : {tf.__version__}')
print(f'Numpy version      : {np.__version__}')
print('Semua library berhasil di-import!')

In [ ]:
# SEL 3: Konfigurasi Global
WINDOW_SIZE = 30       # Jumlah frame per sekuens — HARUS SAMA dengan Fase 1
N_FEATURES  = 63       # 21 landmark tangan x 3 sumbu (x, y, z)
BATCH_SIZE  = 32
EPOCHS      = 150
TEST_SIZE   = 0.15
RANDOM_SEED = 42

print(f'WINDOW_SIZE : {WINDOW_SIZE}')
print(f'N_FEATURES  : {N_FEATURES}')
print(f'Total kolom CSV yang diharapkan: {WINDOW_SIZE * N_FEATURES} kolom')

In [ ]:
# SEL 4: Upload & Load Dataset
# Pastikan file dataset_normal.csv sudah di-upload
# Cara upload: klik ikon Folder (panel kiri) -> klik tombol Upload -> pilih dataset_normal.csv

CSV_FILE = 'dataset_normal.csv'

if not os.path.exists(CSV_FILE):
    print('[ERROR] File dataset_normal.csv tidak ditemukan!')
    print('Silakan upload file tersebut terlebih dahulu.')
else:
    df = pd.read_csv(CSV_FILE)
    print(f'Shape CSV yang dibaca : {df.shape}')
    print(f'Total sekuens         : {df.shape[0]}')
    print(f'Total kolom           : {df.shape[1]}')

    # Reshape dari 2D (N, 1890) ke 3D (N, 30, 63)
    data_3d = df.values.reshape(-1, WINDOW_SIZE, N_FEATURES).astype(np.float32)
    print(f'\nShape 3D: {data_3d.shape}')
    print(f'  - {data_3d.shape[0]} sekuens')
    print(f'  - {data_3d.shape[1]} timestep per sekuens')
    print(f'  - {data_3d.shape[2]} fitur per timestep')

In [ ]:
# SEL 5: Normalisasi dengan MinMaxScaler
# LSTM butuh data dalam skala kecil agar training lebih stabil

data_2d = data_3d.reshape(-1, N_FEATURES)

scaler = MinMaxScaler(feature_range=(0, 1))
data_2d_scaled = scaler.fit_transform(data_2d)

data_scaled = data_2d_scaled.reshape(-1, WINDOW_SIZE, N_FEATURES)

print(f'Nilai min sebelum scaling : {data_2d.min():.4f}')
print(f'Nilai max sebelum scaling : {data_2d.max():.4f}')
print(f'Nilai min setelah scaling : {data_2d_scaled.min():.4f}')
print(f'Nilai max setelah scaling : {data_2d_scaled.max():.4f}')

# Simpan scaler — dibutuhkan di Fase 3!
joblib.dump(scaler, 'scaler_Rehabilitasi.pkl')
print('\nScaler disimpan: scaler_Rehabilitasi.pkl')

In [ ]:
# SEL 6: Split Data Train / Test

X_train_val, X_test = train_test_split(
    data_scaled, test_size=TEST_SIZE, random_state=RANDOM_SEED, shuffle=True
)

print(f'Total data     : {data_scaled.shape[0]} sekuens')
print(f'Data Train+Val : {X_train_val.shape[0]} sekuens ({int((1-TEST_SIZE)*100)}%)')
print(f'Data Test      : {X_test.shape[0]} sekuens ({int(TEST_SIZE*100)}%)')

In [ ]:
# SEL 7: Definisi Arsitektur LSTM-Autoencoder
#
#  Input (30, 63)
#      |
#  [ENCODER]
#  LSTM(128) -> return_sequences=True
#  BatchNormalization()
#  Dropout(0.2)
#  LSTM(64) -> return_sequences=True
#  BatchNormalization()
#  Dropout(0.2)
#  LSTM(32) -> return_sequences=False   <- Bottleneck
#  BatchNormalization()
#      |
#  RepeatVector(30)
#      |
#  [DECODER]
#  LSTM(32) -> return_sequences=True
#  BatchNormalization()
#  Dropout(0.2)
#  LSTM(64) -> return_sequences=True
#  BatchNormalization()
#  Dropout(0.2)
#  LSTM(128) -> return_sequences=True
#  BatchNormalization()
#  TimeDistributed(Dense(63))           <- Output rekonstruksi

model = Sequential(name='LSTM_Autoencoder_Rehabilitasi')

# ENCODER
model.add(LSTM(128, activation='tanh', input_shape=(WINDOW_SIZE, N_FEATURES),
               return_sequences=True, name='encoder_lstm_1'))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(LSTM(64, activation='tanh', return_sequences=True, name='encoder_lstm_2'))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(LSTM(32, activation='tanh', return_sequences=False, name='encoder_lstm_3'))
model.add(BatchNormalization())

# BOTTLENECK
model.add(RepeatVector(WINDOW_SIZE, name='bottleneck'))

# DECODER
model.add(LSTM(32, activation='tanh', return_sequences=True, name='decoder_lstm_1'))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(LSTM(64, activation='tanh', return_sequences=True, name='decoder_lstm_2'))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(LSTM(128, activation='tanh', return_sequences=True, name='decoder_lstm_3'))
model.add(BatchNormalization())

# OUTPUT
model.add(TimeDistributed(Dense(N_FEATURES), name='output_layer'))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

model.summary()

In [ ]:
# SEL 8: Training Model

callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ModelCheckpoint('otak_Rehabilitasi_best.keras', monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1)
]

print('=== Training Dimulai ===')
history = model.fit(
    X_train_val, X_train_val,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)
print('=== Training Selesai ===')

In [ ]:
# SEL 9: Visualisasi Loss Training

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='Training Loss',   color='royalblue', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', color='tomato',    linewidth=2)
axes[0].set_title('MSE Loss per Epoch', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'],     label='Training MAE',   color='royalblue', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', color='tomato',    linewidth=2)
axes[1].set_title('MAE per Epoch', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Kurva Pelatihan LSTM-Autoencoder', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan: training_loss.png')

In [ ]:
# SEL 10: Kalkulasi Threshold Statistik (WAJIB!)

X_pred_train = model.predict(X_train_val, verbose=1)

mse_train = np.mean(np.power(X_train_val - X_pred_train, 2), axis=(1, 2))

mu_mse     = np.mean(mse_train)
sigma_mse  = np.std(mse_train)
THRESHOLD  = mu_mse + (4 * sigma_mse)

print('\n' + '='*55)
print('   HASIL KALKULASI THRESHOLD ANOMALI')
print('='*55)
print(f'   Jumlah data training  : {len(mse_train)}')
print(f'   Rata-rata MSE (mu)    : {mu_mse:.10f}')
print(f'   Std Dev MSE (sigma)   : {sigma_mse:.10f}')
print(f'   Min MSE               : {mse_train.min():.10f}')
print(f'   Max MSE               : {mse_train.max():.10f}')
print('='*55)
print(f'\n   *** THRESHOLD FINAL = {THRESHOLD:.10f} ***')
print('='*55)

# Simpan ke file teks
with open('threshold_value.txt', 'w') as f:
    f.write(f'THRESHOLD = {THRESHOLD:.10f}\n')
    f.write(f'mu_mse    = {mu_mse:.10f}\n')
    f.write(f'sigma_mse = {sigma_mse:.10f}\n')
print('\nNilai threshold disimpan: threshold_value.txt')

In [ ]:
# SEL 11: Visualisasi Distribusi MSE

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(mse_train, bins=50, color='steelblue', alpha=0.75, edgecolor='white', label='MSE Training')
axes[0].axvline(THRESHOLD, color='red',   linestyle='--', linewidth=2.5, label=f'Threshold = {THRESHOLD:.5f}')
axes[0].axvline(mu_mse,    color='green', linestyle='--', linewidth=1.5, label=f'Mean = {mu_mse:.5f}')
axes[0].set_title('Distribusi Reconstruction Error (MSE)\nData Normal', fontsize=12, fontweight='bold')
axes[0].set_xlabel('MSE Error')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].boxplot(mse_train, vert=True, patch_artist=True, boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].axhline(THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Threshold = {THRESHOLD:.5f}')
axes[1].set_title('Box Plot MSE Error\nData Normal', fontsize=12, fontweight='bold')
axes[1].set_ylabel('MSE Error')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Analisis Distribusi Reconstruction Error', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('distribusi_mse.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik distribusi MSE disimpan: distribusi_mse.png')

In [ ]:
# SEL 12: Evaluasi pada Test Set

X_pred_test = model.predict(X_test, verbose=1)
mse_test    = np.mean(np.power(X_test - X_pred_test, 2), axis=(1, 2))

y_pred_test = (mse_test >= THRESHOLD).astype(int)
n_correct   = np.sum(y_pred_test == 0)
n_total     = len(y_pred_test)
accuracy    = n_correct / n_total * 100

print(f'Evaluasi pada Test Set (semuanya data Normal):')
print(f'  Total sekuens test      : {n_total}')
print(f'  Diklasifikasi NORMAL    : {n_correct} ({accuracy:.1f}%)')
print(f'  False Positive          : {n_total - n_correct} ({100-accuracy:.1f}%)')
print(f'  Rata-rata MSE test      : {mse_test.mean():.8f}')

In [ ]:
# SEL 13: Simpan Model Final

model.save('otak_Rehabilitasi.keras')

print('=' * 55)
print('  SEMUA FILE BERHASIL DISIMPAN:')
print('=' * 55)
print('  otak_Rehabilitasi.keras       - Model AI')
print('  scaler_Rehabilitasi.pkl    - Normalisasi data')
print('  threshold_value.txt       - Nilai threshold')
print('  training_loss.png         - Grafik training')
print('  distribusi_mse.png        - Grafik MSE')
print('=' * 55)
print('\nJalankan SEL 14 untuk download semua file otomatis!')

In [ ]:
# SEL 14: Download Semua File ke Komputer Anda

from google.colab import files

print('Memulai download...')
files.download('otak_Rehabilitasi.keras')
files.download('scaler_Rehabilitasi.pkl')
files.download('threshold_value.txt')
files.download('training_loss.png')
files.download('distribusi_mse.png')

print('\nSetelah download selesai:')
print('1. Pindahkan otak_Rehabilitasi.keras  -> folder models/')
print('2. Pindahkan scaler_Rehabilitasi.pkl -> folder models/')
print('3. Pindahkan threshold_value.txt  -> folder models/')
print('4. Jalankan: python fase3_Rehabilitasi_app.py')